In [ ]:
import sys, os
os.chdir(os.path.abspath('..'))
sys.path.insert(0, os.getcwd())

### sids that are in Nymph family with genera on Lepid tree but not on Nymph tree (this was previously used for a pairwise self-similarity test)

In [ ]:
from utils.utils import paths, load_pickle
from preprocessing.lepid.phylo import build_tree_lepid
from preprocessing.nymph.phylo import build_tree_nymph


tree_lepid = build_tree_lepid()
tree_nymph = build_tree_nymph()

sids_lepid = [tip.name for tip in tree_lepid.get_terminals()]
sids_nymph = [tip.name for tip in tree_nymph.get_terminals()]

class_data_lepid = load_pickle(paths["metadata"]["lepid"] / "class_data.pkl")

genera_lepid = set([sid.split("_")[0] for sid in sids_lepid])
genera_nymph = set([sid.split("_")[0] for sid in sids_nymph])
genera_lepid_non_nymph = sorted(genera_lepid - genera_nymph)

sids_genera_lepid_non_nymph_nymphalidae = set()
for sid in class_data_lepid.keys():
    if class_data_lepid[sid]["family"] == "nymphalidae":
        if sid.split("_")[0] in genera_lepid_non_nymph:
            sids_genera_lepid_non_nymph_nymphalidae.add(sid)

In [ ]:
[sid for sid in sids_lepid if sid in sids_genera_lepid_non_nymph_nymphalidae]

In [ ]:
from itertools import combinations

from utils.phylo import get_tree
from preprocessing.lepid.phylo import build_tree_lepid
from preprocessing.nymph.phylo import build_tree_nymph
from utils.utils import paths, load_pickle
from preprocessing.lepid.phylo import combine_trees_lepid_nymph
from preprocessing.common.phylo import prune_tree, augment_class_data

In [ ]:
def print_pairwise_distances(tree, labels):
    for a, b in combinations(labels, 2):
        print(f"tree.distance('{a}', '{b}') = {tree.distance(a, b)}")

def sids_starting_with(sids, prefix):
    return sorted([sid for sid in sids if sid.startswith(prefix)])

def labels_in_sids(sids, labels, header=None):
    if header:
        print(header)
    for sid in labels:
        print(sid in sids)

def sids_in_rank(sid, rank, class_data):
    rank_name = class_data[sid][rank]
    return sorted([sid for sid in class_data.keys() if class_data[sid][rank] == rank_name])

def neighbor_genus_dists(sid, tree, class_data, remove_genus_sid=True):

    sids_tree = {tip.name for tip in tree.get_terminals()}

    cd_sid = class_data[sid]
    rank = "family"
    if cd_sid["subfamily"] is not None:
        rank = "subfamily"
    if cd_sid["tribe"] is not None:
        rank = "tribe"

    print("Lowestest rank above genus:", rank)

    sids_cands = set(sids_in_rank(sid, rank, class_data)) & sids_tree
    if remove_genus_sid:
        sids_cands = {sid for sid in sids_cands if class_data[sid]["genus"] != cd_sid["genus"]}

    for sid_cand in sorted(sids_cands):
        print(f"{sid_cand}: {tree.distance(sid, sid_cand)}")

def closest_species_outside_genus(sid, tree, class_data):

    sids_tree = {tip.name for tip in tree.get_terminals()}

    cd_sid = class_data[sid]
    rank = "family"
    if cd_sid["subfamily"] is not None:
        rank = "subfamily"
    if cd_sid["tribe"] is not None:
        rank = "tribe"

    print("Lowestest rank above genus:", rank)

    sids_cands = set(sids_in_rank(sid, rank, class_data)) & sids_tree
    sids_cands = {sid for sid in sids_cands if class_data[sid]["genus"] != cd_sid["genus"]}

    sid_closest = None
    dist_closest = float("inf")
    for sid_cand in sorted(sids_cands):
        dist = tree.distance(sid, sid_cand)
        if dist < dist_closest:
            sid_closest = sid_cand
            dist_closest = dist

    print(f"Closest species outside genus: {sid_closest} (distance {dist_closest:.2f})")

In [ ]:
tree = get_tree(dataset="lepid")  # merged
sids_tree = {tip.name for tip in tree.get_terminals()}

tree_lepid = build_tree_lepid()
tree_nymph = build_tree_nymph()
sids_lepid = {tip.name for tip in tree_lepid.get_terminals()}
sids_nymph = {tip.name for tip in tree_nymph.get_terminals()}

class_data = load_pickle(paths["metadata"]["lepid"] / "class_data.pkl")
class_data_aug = augment_class_data(class_data, tree)

### closest genus to "colias" genus: "zerene"

In [ ]:
neighbor_genus_dists('colias_palaeno', tree, class_data_aug, remove_genus_sid=False)

### sid-combo distances

In [ ]:
labels = ['adelpha_mesentina', 'aglais_urticae', 'aglais_io', 'daedalma_dinias']
print_pairwise_distances(tree, labels)